In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

def test_www_convention(urls, y, dataset_name):
    """y must be 1=phishing, 0=legitimate. urls is a Series of raw URL strings."""
    urls = urls.astype(str)
    has_https = urls.str.startswith('https://')
    starts_www = urls.str.startswith('https://www.')

    pred_https = (~has_https).astype(int)
    pred_www = (~starts_www).astype(int)

    print(f"=== {dataset_name} (n={len(urls)}, phishing={int(y.sum())}, legit={int((y==0).sum())}) ===")
    print(f"IsHTTPS-only baseline    acc={accuracy_score(y, pred_https):.4f}  f1={f1_score(y, pred_https):.4f}")
    print(f"https://www. baseline    acc={accuracy_score(y, pred_www):.4f}  f1={f1_score(y, pred_www):.4f}")
    print()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/Phishing_Analysis'

Mounted at /content/drive


In [ ]:
# PhishOFE (Kustiawan & Ghauth)

phishofe_full = pd.read_csv(f'{base_path}/PhishOFE_ds.csv')

assert {'URL', 'label'}.issubset(phishofe_full.columns), phishofe_full.columns.tolist()
n_raw = len(phishofe_full)
phishofe_full = phishofe_full.dropna(subset=['URL', 'label'])
print(f'Rows: {n_raw:,} raw -> {len(phishofe_full):,} after dropping NaN URL/label')

y_phishofe_full = phishofe_full['label'].astype(int)

# label-orientation check: legitimate URLs should be the ones starting https://www.
print('\nlabel vs https://www. prefix (rows):')
print(pd.crosstab(phishofe_full['URL'].astype(str).str.startswith('https://www.'),
                  y_phishofe_full, rownames=['starts_www'], colnames=['label']))
print('label==1 share:', f'{y_phishofe_full.mean():.4f}', '(should be the phishing class)\n')

test_www_convention(phishofe_full['URL'], y_phishofe_full, 'PhishOFE (Drive copy, full)')

=== PhishOFE (n=98245, phishing=46673, legit=51572) ===
IsHTTPS-only baseline    acc=0.5515  f1=0.1058
https://www. baseline    acc=0.9882  f1=0.9874



In [ ]:
# Mendeley "Phishing URL Dataset" (Kaitholikkal & Arthi B, 2024)
mendeley1 = pd.read_csv(f'{base_path}/mendeley1_url.csv')
y_mendeley1 = (mendeley1['type'] == 'phishing').astype(int)
test_www_convention(mendeley1['url'], y_mendeley1, 'Mendeley Phishing URL Dataset')

# Mendeley "LegitPhish" (Potpelwar, Kulkarni & Waghmare, 2025)
legitphish = pd.read_csv(f'{base_path}/legitphish_url.csv').dropna(subset=['ClassLabel', 'URL'])
y_legitphish = 1 - legitphish['ClassLabel'].astype(int)   # ClassLabel: 1=legit,0=phishing -> flip
test_www_convention(legitphish['URL'], y_legitphish, 'Mendeley LegitPhish')

# Mendeley "PhishLegitURLs" (Potpelwar, 2024)
phishlegit = pd.read_excel(f'{base_path}/phishlegit_url.xlsx')
y_phishlegit = 1 - phishlegit['ClassLabel'].astype(int)   # ClassLabel: 1=legit,0=phishing -> flip
test_www_convention(phishlegit['URL'], y_phishlegit, 'Mendeley PhishLegitURLs')

=== Mendeley Phishing URL Dataset (n=450176, phishing=104438, legit=345738) ===
IsHTTPS-only baseline    acc=0.9855  f1=0.9678
https://www. baseline    acc=0.9967  f1=0.9929

=== Mendeley LegitPhish (n=101218, phishing=63678, legit=37540) ===
IsHTTPS-only baseline    acc=0.9651  f1=0.9715
https://www. baseline    acc=0.9947  f1=0.9958

=== Mendeley PhishLegitURLs (n=905, phishing=699, legit=206) ===
IsHTTPS-only baseline    acc=0.7746  f1=0.8291
https://www. baseline    acc=0.7713  f1=0.8709



In [ ]:
# Feng & Yue, Colorado School of Mines (SACMAT 2020)

import urllib.request, dill

feng_yue_url = 'https://www.dropbox.com/scl/fi/gs6qqw2rovclnkhsdqslt/vonDataset20180426.dill?rlkey=s262zhfhottvxmdl41tz969g9&dl=1'
urllib.request.urlretrieve(feng_yue_url, 'vonDataset.dill')   # large download, may take a few minutes

with open('vonDataset.dill', 'rb') as f:
    von_data = dill.load(f)

int_to_char = {v: k for k, v in von_data['char_to_int'].items()}

def decode_prefix(row, n=20):
    return ''.join(int_to_char.get(i, '') for i in row[:n] if i != 0)

import numpy as np
X_von = np.concatenate([von_data['train_x'], von_data['val_x'], von_data['test_x']], axis=0)
y_von = np.concatenate([von_data['train_y'], von_data['val_y'], von_data['test_y']], axis=0)

prefixes_von = pd.Series([decode_prefix(row) for row in X_von])
test_www_convention(prefixes_von, pd.Series(y_von), 'Feng & Yue (Colorado School of Mines)')

=== Feng & Yue (Colorado School of Mines) (n=1559362, phishing=759362, legit=800000) ===
IsHTTPS-only baseline    acc=0.4961  f1=0.6577
https://www. baseline    acc=0.4917  f1=0.6569

